Prerequisite: Download "data/raw/inferlink_extraction_v3" from ta2 github repo.

Process Inferlink's v3 extraction into a similar file as the ground truth file.

In [24]:
import json
import os

import pandas as pd

from agent_k.config.schemas import InferlinkEvalColumns


def get_nested(data, path, default=None):
    """Helper function to get a nested value from a dictionary."""
    try:
        for key in path:
            if isinstance(data, dict):
                data = data.get(key, default)
            elif isinstance(data, list) and isinstance(key, int):
                data = data[key]
            else:
                return default
        return data
    except (KeyError, IndexError, TypeError):
        return default


# Get all JSON files in the directory
json_dir = "data/raw/inferlink_extraction_v3/"
json_files = [
    os.path.join(json_dir, f) for f in os.listdir(json_dir) if f.endswith(".json")
]

df_gt = pd.read_csv("data/processed/ground_truth/inferlink_ground_truth.csv")
# Get unique pairs of record_id and main_commodity
id_main_commodity = (
    df_gt[
        [
            InferlinkEvalColumns.CDR_RECORD_ID.value,
            InferlinkEvalColumns.MAIN_COMMODITY.value,
        ]
    ]
    .drop_duplicates()
    .values.tolist()
)
gt_record_ids = [id for id, _ in id_main_commodity]
record_id_main_commodity = {
    id: main_commodity for id, main_commodity in id_main_commodity
}

matched_data = {}
for file_path in json_files:
    try:
        with open(file_path, "r") as f:
            data = json.load(f)
            for d in data:
                if d.get("record_id", "") in gt_record_ids:
                    matched_data[d["record_id"]] = d
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

rows = []
for record_id, d in matched_data.items():
    metadata = {}
    metadata["cdr_record_id"] = record_id
    metadata["mineral_site_name"] = get_nested(d, ["name"])
    metadata["country"] = get_nested(
        d, ["location_info", "country", 0, "observed_name"]
    )
    metadata["state_or_province"] = get_nested(
        d, ["location_info", "state_or_province", 0, "observed_name"]
    )
    metadata["main_commodity"] = record_id_main_commodity[record_id]

    if "mineral_inventory" in d:
        for mi in d["mineral_inventory"]:
            row = metadata.copy()
            row["commodity_observed_name"] = get_nested(
                mi, ["commodity", "observed_name"]
            )
            row["category_observed_name"] = get_nested(
                mi, ["category", 0, "observed_name"]
            )
            row["ore_unit_observed_name"] = get_nested(
                mi, ["ore", "unit", "observed_name"]
            )
            row["ore_value"] = get_nested(mi, ["ore", "value"])
            row["grade_unit_observed_name"] = get_nested(
                mi, ["grade", "unit", "observed_name"]
            )
            row["grade_value"] = get_nested(mi, ["grade", "value"])
            row["zone"] = get_nested(mi, ["zone"])
            rows.append(row)
    else:
        row = metadata.copy()
        rows.append(row)


# convert to csv
df_inferlink_extraction_v3 = pd.DataFrame(rows)
df_inferlink_extraction_v3.head()

,cdr_record_id,mineral_site_name,country,state_or_province,main_commodity,commodity_observed_name,category_observed_name,ore_unit_observed_name,ore_value,grade_unit_observed_name,grade_value,zone
0,021ee3d3ec6928901d9427796372edaada0d332f15d030...,Red Chris Deposit,Canada,British Columbia,copper,copper,measured+indicated,None,619417.0,None,0.38,None
1,021ee3d3ec6928901d9427796372edaada0d332f15d030...,Red Chris Deposit,Canada,British Columbia,copper,copper,inferred,None,619129.0,None,0.30,None
2,021ee3d3ec6928901d9427796372edaada0d332f15d030...,Red Chris Deposit,Canada,British Columbia,copper,copper,measured+indicated,None,489151.0,None,0.43,None
3,021ee3d3ec6928901d9427796372edaada0d332f15d030...,Red Chris Deposit,Canada,British Columbia,copper,copper,inferred,None,437939.0,None,0.36,None
4,021ee3d3ec6928901d9427796372edaada0d332f15d030...,Red Chris Deposit,Canada,British Columbia,copper,copper,measured+indicated,None,312571.0,None,0.54,None


In [25]:
from agent_k.setup.construct_inferlink_eval import (
    normalize_grade_units,
    normalize_ore_units,
)

# Assign default units if missing
df_inferlink_extraction_v3["ore_unit_observed_name"] = df_inferlink_extraction_v3[
    "ore_unit_observed_name"
].fillna("tonnes")
df_inferlink_extraction_v3["grade_unit_observed_name"] = df_inferlink_extraction_v3[
    "grade_unit_observed_name"
].fillna("percent")

# Assign default values if missing
df_inferlink_extraction_v3["ore_value"] = df_inferlink_extraction_v3[
    "ore_value"
].fillna(0)
df_inferlink_extraction_v3["grade_value"] = df_inferlink_extraction_v3[
    "grade_value"
].fillna(0)

master_metadata_df = df_inferlink_extraction_v3.loc[
    :, ["cdr_record_id", "mineral_site_name", "country", "state_or_province"]
].drop_duplicates()
master_inventory_df = df_inferlink_extraction_v3.loc[
    :,
    [
        "cdr_record_id",
        "main_commodity",
        "commodity_observed_name",
        "category_observed_name",
        "ore_unit_observed_name",
        "ore_value",
        "grade_unit_observed_name",
        "grade_value",
        "zone",
    ],
]
master_inventory_df = normalize_ore_units(master_inventory_df)
master_inventory_df = normalize_grade_units(master_inventory_df)

Error normalizing ore units for row 377: Unit 'percent' not recognized
Error normalizing ore units for row 378: Unit 'percent' not recognized
Error normalizing ore units for row 1051: Unit 'metres' not recognized
Error normalizing ore units for row 1052: Unit 'metres' not recognized
Error normalizing ore units for row 1053: Unit 'metres' not recognized
Error normalizing ore units for row 1054: Unit 'metres' not recognized
Error normalizing ore units for row 1055: Unit 'metres' not recognized
Error normalizing grade units for row 434: Grade unit '% WO 3' not recognized
Error normalizing grade units for row 435: Grade unit '% WO 3' not recognized
Error normalizing grade units for row 665: Grade unit 'WO3%' not recognized
Error normalizing grade units for row 666: Grade unit 'WO3%' not recognized
Error normalizing grade units for row 667: Grade unit 'WO3%' not recognized
Error normalizing grade units for row 668: Grade unit 'WO3%' not recognized
Error normalizing grade units for row 669: 

In [26]:
cols_to_drop = [
    "ore_unit_observed_name",
    "grade_unit_observed_name",
    "ore_value",
    "grade_value",
    "zone",
]
master_inventory_df = master_inventory_df.drop(columns=cols_to_drop)

category_to_resource_or_reserve = {
    "inferred": "resource",
    "measured": "resource",
    "indicated": "resource",
    "mineral resource": "resource",
    "measured+indicated": "resource",  # Exception
    "proven+probable": "reserve",  # Exception
    "proved": "reserve",
    "probable": "reserve",
    "proven": "reserve",
    "mineralresource": "resource",
}
# remove any whitespace from category (e.g. "proven + probable" -> "proven+probable")
master_inventory_df["category_observed_name"] = (
    master_inventory_df["category_observed_name"].str.replace(" ", "").str.lower()
)

master_inventory_df.insert(
    4,
    "resource_or_reserve",
    master_inventory_df["category_observed_name"].map(category_to_resource_or_reserve),
)
# Assert for all rows have "category_observed_name" it has a corresponding "resource_or_reserve" value
assert (
    not master_inventory_df[
        master_inventory_df["category_observed_name"].notna()
        & master_inventory_df["resource_or_reserve"].isna()
    ]
    .any()
    .any()
)
master_inventory_df.drop(columns=["category_observed_name"], inplace=True)

master_inventory_df["contained_metal"] = (
    master_inventory_df["normalized_ore_value"]
    * master_inventory_df["normalized_grade_value"]
    / 100
)

# Note: mineral site with no inventory data will have a NaN value in the "category_observed_name" column
# and thus a NaN value in the "resource_or_reserve" column. These rows will be dropped during the groupby operation.

# Group by cdr_record_id, resource_or_reserve and sum the normalized_ore_value
resource_n_reserve_total_tonnage = (
    master_inventory_df.groupby(
        [
            InferlinkEvalColumns.CDR_RECORD_ID.value,
            InferlinkEvalColumns.MAIN_COMMODITY.value,
            InferlinkEvalColumns.COMMODITY_OBSERVED_NAME.value,
            "resource_or_reserve",
        ]
    )
    .agg(
        {
            "normalized_ore_value": "sum",
            "contained_metal": "sum",
        }
    )
    .reset_index()
)

# pivot the resource_n_reserve column to wide format
resource_n_reserve_total_tonnage = resource_n_reserve_total_tonnage.pivot(
    index=[
        InferlinkEvalColumns.CDR_RECORD_ID.value,
        InferlinkEvalColumns.MAIN_COMMODITY.value,
        InferlinkEvalColumns.COMMODITY_OBSERVED_NAME.value,
    ],
    columns="resource_or_reserve",
    values=["normalized_ore_value", "contained_metal"],
)
# expand the index to separate columns
resource_n_reserve_total_tonnage.columns = [
    f"{col[0]}_{col[1]}" for col in resource_n_reserve_total_tonnage.columns
]
resource_n_reserve_total_tonnage = resource_n_reserve_total_tonnage.reset_index()

# Left join master_metadata_df (mineral site name, country, state or province) to
# master_inventory_df (resource or reserve, total tonnage, contained metal) on cdr_record_id
master_df = pd.merge(
    resource_n_reserve_total_tonnage,
    master_metadata_df,
    on=InferlinkEvalColumns.CDR_RECORD_ID.value,
    how="left",
)

cols_to_rename = {
    "normalized_ore_value_resource": InferlinkEvalColumns.TOTAL_MINERAL_RESOURCE_TONNAGE.value,
    "normalized_ore_value_reserve": InferlinkEvalColumns.TOTAL_MINERAL_RESERVE_TONNAGE.value,
    "contained_metal_resource": InferlinkEvalColumns.TOTAL_MINERAL_RESOURCE_CONTAINED_METAL.value,
    "contained_metal_reserve": InferlinkEvalColumns.TOTAL_MINERAL_RESERVE_CONTAINED_METAL.value,
    "country_observed_name": InferlinkEvalColumns.COUNTRY.value,
    "state_or_province_observed_name": InferlinkEvalColumns.STATE_OR_PROVINCE.value,
    "mining_name": InferlinkEvalColumns.MINERAL_SITE_NAME.value,
}
master_df = master_df.rename(columns=cols_to_rename)
master_df.head()

,cdr_record_id,main_commodity,commodity_observed_name,total_mineral_reserve_tonnage,total_mineral_resource_tonnage,total_mineral_reserve_contained_metal,total_mineral_resource_contained_metal,mineral_site_name,country,state_or_province
0,0200a1c6d2cfafeb485d815d95966961d4c119e8662b8b...,nickel,cobalt,0.428563,1172.657486,0.000047,0.126648,TURNAGAIN NICKEL PROJECT,Canada,British Columbia
1,0200a1c6d2cfafeb485d815d95966961d4c119e8662b8b...,nickel,nickel,0.428563,1173.828972,0.000729,1.933819,TURNAGAIN NICKEL PROJECT,Canada,British Columbia
2,0204fc707f5b1944308624520cd422c4f1cb478046f664...,tungsten,tungsten,7.325000,15.322878,0.024654,0.043579,Los Santos Mine Project,Spain,Salamanca
3,020548e1aca4c1ca222149e11a79f15bd7594c02eb1216...,earth_metals,dysprosium,NaN,0.000041,NaN,0.000000,Crater Lake Project,Canada,Quebec
4,020548e1aca4c1ca222149e11a79f15bd7594c02eb1216...,earth_metals,lanthanum,NaN,0.000041,NaN,0.000000,Crater Lake Project,Canada,Quebec


In [27]:
master_df = master_df.merge(
    df_gt[["id", "cdr_record_id", "main_commodity", "commodity_observed_name"]],
    on=[
        InferlinkEvalColumns.CDR_RECORD_ID.value,
        InferlinkEvalColumns.MAIN_COMMODITY.value,
        InferlinkEvalColumns.COMMODITY_OBSERVED_NAME.value,
    ],
    how="left",
)
# Move id to the first column
master_df = master_df[["id", *master_df.columns[:-1]]]
master_df.head()

,id,cdr_record_id,main_commodity,commodity_observed_name,total_mineral_reserve_tonnage,total_mineral_resource_tonnage,total_mineral_reserve_contained_metal,total_mineral_resource_contained_metal,mineral_site_name,country,state_or_province
0,1.0,0200a1c6d2cfafeb485d815d95966961d4c119e8662b8b...,nickel,cobalt,0.428563,1172.657486,0.000047,0.126648,TURNAGAIN NICKEL PROJECT,Canada,British Columbia
1,2.0,0200a1c6d2cfafeb485d815d95966961d4c119e8662b8b...,nickel,nickel,0.428563,1173.828972,0.000729,1.933819,TURNAGAIN NICKEL PROJECT,Canada,British Columbia
2,3.0,0204fc707f5b1944308624520cd422c4f1cb478046f664...,tungsten,tungsten,7.325000,15.322878,0.024654,0.043579,Los Santos Mine Project,Spain,Salamanca
3,4.0,020548e1aca4c1ca222149e11a79f15bd7594c02eb1216...,earth_metals,dysprosium,NaN,0.000041,NaN,0.000000,Crater Lake Project,Canada,Quebec
4,5.0,020548e1aca4c1ca222149e11a79f15bd7594c02eb1216...,earth_metals,lanthanum,NaN,0.000041,NaN,0.000000,Crater Lake Project,Canada,Quebec


In [28]:
master_df.to_csv("data/processed/inferlink_extraction_v3.csv", index=False)